In [1]:
import os

current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, os.pardir))
os.chdir(parent_dir)
print("Current working directory:", os.getcwd())

Current working directory: /Users/alvarovilladangos/Desktop/TradingBot/Forex


In [ ]:
from ibkr.app import IBApp
from ibkr.contract import option, combo_leg, spread
from ibkr.order import market, BUY, SELL

app = IBApp("127.0.0.1", 4002, client_id=79, account="DUE310707")


ERROR -1 2104 La conexión al centro de datos funciona correctamente:cashfarm
ERROR -1 2104 La conexión al centro de datos funciona correctamente:usfarm
ERROR -1 2107 La conexión al centro de datos HMDS está inactiva pero debería estar disponible bajo demanda.ushmds
ERROR -1 2158 La conexión de la granja de datos "sec-def" funciona correctamente:secdefnj


In [ ]:
from typing import Callable
from ibapi.contract import Contract
from ibapi.order import Order
from ibapi.client import EClient
from ibapi.wrapper import EWrapper
from ibapi.common import *
import time
import threading

class IBApp(EWrapper, EClient):
    def __init__(self, ip: str, port: int, client_id: int, account: str):
        EWrapper.__init__(self)
        EClient.__init__(self, self)
        self.account = account
        self.resolved_contract = None
        self.connect(ip, port, client_id)
        threading.Thread(target=self.run, daemon=True).start()

    def order_target_value(self, contract: Contract, order_type: Callable[..., Order], target: float, **kwargs) -> int:
        """
        Adjust a position to a target value in the portfolio.

        This method places an order to modify a position such that its value matches
        the specified target value. If the position does not exist, this is equivalent
        to placing a new order for the target value. If the position exists, it calculates
        the difference between the target value and the current position value, then
        places an order for the difference.

        Args:
            contract (Contract): The IB Contract object specifying the financial instrument 
                                 (e.g., stock, option, etc.) for which the order is being placed.
            order_type (Callable[..., Order]): A function or callable that creates an order object 
                                               (e.g., market, limit, stop). Must accept `action`, `quantity`,
                                               and other keyword arguments.
            target (float): The desired target value of the position in the portfolio's currency.
            **kwargs: Additional keyword arguments to pass to the `order_type` callable. 
                      Common arguments include price for limit orders or stop price for stop orders.

        Returns:
            int: The order ID of the newly created order.

        How it works:
            1. Calls `_calculate_order_value_quantity` to determine the quantity
               of the asset required to meet the target value.
            2. Calls `_calculate_order_target_quantity` to adjust the quantity based
               on the difference between the target and current values.
            3. Creates an order object using `order_type`, specifying:
               - `action`: Either `BUY` or `SELL` depending on whether the position needs to be increased or decreased.
               - `quantity`: The absolute value of the calculated quantity.
               - Additional keyword arguments provided via `**kwargs`.
            4. Sends the order using `send_order`.

        Notes:
            - The `target` value should be calculated in the portfolio's base currency.
            - The `order_type` callable should create and return a valid IB Order object.

        Example:
            To set a stock position to a value of $10,000 using a market order:

            ```python
            contract = Contract()
            contract.symbol = "AAPL"
            contract.secType = "STK"
            contract.exchange = "SMART"
            contract.currency = "USD"

            app.order_target_value(contract, MarketOrder, target=10000)
            ```
        """
        target_quantity: int = self._calculate_order_value_quantity(contract, target)
        quantity: int = self._calculate_order_target_quantity(contract, target_quantity)
        order: Order = order_type(
            action="SELL" if quantity < 0 else "BUY", quantity=abs(quantity), **kwargs
        )
        return self.send_order(contract, order)

    def _calculate_order_value_quantity(self, contract: Contract, target: float) -> int:
        """
        Placeholder for method to calculate the quantity needed to reach the target value.

        Args:
            contract (Contract): The financial instrument for which to calculate the quantity.
            target (float): The desired target value of the position in portfolio currency.

        Returns:
            int: The quantity of the asset required to meet the target value.
        """
        pass  # Replace with actual implementation

    def _calculate_order_target_quantity(self, contract: Contract, target_quantity: int) -> int:
        """
        Placeholder for method to calculate the final target quantity.

        Args:
            contract (Contract): The financial instrument for which to calculate the quantity.
            target_quantity (int): The preliminary target quantity calculated based on the target value.

        Returns:
            int: The adjusted quantity of the asset required.
        """
        pass  # Replace with actual implementation

    def send_order(self, contract: Contract, order: Order) -> int:
        """
        Placeholder for method to send an order to the IB API.

        Args:
            contract (Contract): The financial instrument associated with the order.
            order (Order): The order to be sent.

        Returns:
            int: The order ID of the newly placed order.
        """
        pass  # Replace with actual implementation

    def resolve_contract(self, contract: Contract, request_id: int = 1) -> Contract:
        """
        Resolve contract details using the IB API and return the resolved contract.

        Args:
            contract (Contract): The contract to resolve.
            request_id (int, optional): The ID of the request for contract resolution.

        Returns:
            Contract: The resolved contract.

        Raises:
            ValueError: If the contract could not be resolved.
        """
        self.resolved_contract = None  # Reset resolved contract
        self.reqContractDetails(request_id, contract)  # Send request
        time.sleep(2)  # Wait for callbacks to process response
        if self.resolved_contract is None:
            raise ValueError("The contract could not be resolved.")
        return self.resolved_contract

    def contractDetails(self, reqId, contractDetails):
        """
        Callback that handles contract details received from the IB API.

        Args:
            reqId (int): Unique ID for the contract details request.
            contractDetails (ContractDetails): Details of the resolved contract.
        """
        print(f"Received contract details for reqId {reqId}: {contractDetails}")
    
        self.resolved_contract = contractDetails.contract

    def contractDetailsEnd(self, reqId):
        """
        Callback that signals the end of contract details transmission.

        Args:
            reqId (int): Unique ID for the contract details request.
        """
        print(f"Finished receiving contract details for request ID: {reqId}")


In [25]:

app = IBApp("127.0.0.1", 4002, client_id=32424, account="DUE310707")
long_call_contract = option("TSLA", "SMART", "202503", 260, "CALL")
resolved_contract = app.resolve_contract(long_call_contract)
print(resolved_contract)


ERROR -1 2104 La conexión al centro de datos funciona correctamente:cashfarm
ERROR -1 2104 La conexión al centro de datos funciona correctamente:usfarm
ERROR -1 2107 La conexión al centro de datos HMDS está inactiva pero debería estar disponible bajo demanda.ushmds
ERROR -1 2158 La conexión de la granja de datos "sec-def" funciona correctamente:secdefnj


ValueError: The contract could not be resolved.

In [18]:
app.reqContractDetails(reqId=44,contract=long_call_contract)

In [19]:
app.contractDetails(44, contractDetails)

NameError: name 'contractDetails' is not defined

In [15]:
app.contractDetails(reqId=44)

TypeError: EWrapper.contractDetails() missing 1 required positional argument: 'contractDetails'

In [8]:
app.resolved_contract

AttributeError: 'IBApp' object has no attribute 'resolved_contract'

In [11]:

long_call_contract = option("TSLA", "SMART", "202503", 260, "CALL")
long_call = app.resolve_contract(long_call_contract)

long_put_contract = option("TSLA", "SMART", "202404", 260, "PUT")
long_put = app.resolve_contract(long_put_contract)

leg_1 = combo_leg(long_call, 1, BUY)
leg_2 = combo_leg(long_put, 1, BUY)

long_strangle = spread([leg_1, leg_2])

order = market(BUY, 1)
app.send_order(long_strangle, order)



AttributeError: 'IBApp' object has no attribute 'resolved_contract'

In [ ]:

short_call = app.resolve_contract(option("TSLA", "SMART", "202503", 295, "CALL"))
long_call = app.resolve_contract(option("TSLA", "SMART", "202503", 305, "CALL"))
short_put = app.resolve_contract(option("TSLA", "SMART", "202503", 240, "PUT"))
long_put = app.resolve_contract(option("TSLA", "SMART", "202503", 230, "PUT"))

leg_1 = combo_leg(short_call, 1, SELL)
leg_2 = combo_leg(long_call, 1, BUY)
leg_3 = combo_leg(short_put, 1, SELL)
leg_4 = combo_leg(long_put, 1, BUY)

short_iron_condor = spread([leg_1, leg_2, leg_3, leg_4])

order = market(BUY, 1)
app.send_order(short_iron_condor, order)

app.disconnect()